# Soma máxima de subarray contíguo

## Divisão e conquista

In [1]:
import pandas as pd

class ContadorSubarray:
    def __init__(self):
        self.chamadas = 0
        self.passos = []


def max_subarray(A, inicio=0, fim=None, contador=None):
    if fim is None:
        fim = len(A) - 1
    if contador is None:
        contador = ContadorSubarray()

    contador.chamadas += 1

    if inicio == fim:
        contador.passos.append({
            'chamada': f'MAX_SUBARRAY({inicio}, {fim})',
            'subarray': [A[inicio]],
            'melhor_soma': A[inicio],
            'tipo': 'caso base'
        })
        return inicio, fim, A[inicio], contador

    meio = (inicio + fim) // 2

    esq_i, esq_j, soma_esq, contador = max_subarray(A, inicio, meio, contador)
    dir_i, dir_j, soma_dir, contador = max_subarray(A, meio + 1, fim, contador)
    cruz_i, cruz_j, soma_cruz = max_subarray_cruzando(A, inicio, meio, fim)

    if soma_esq >= soma_dir and soma_esq >= soma_cruz:
        resposta = (esq_i, esq_j, soma_esq, 'esquerda')
    elif soma_dir >= soma_esq and soma_dir >= soma_cruz:
        resposta = (dir_i, dir_j, soma_dir, 'direita')
    else:
        resposta = (cruz_i, cruz_j, soma_cruz, 'cruzando')

    contador.passos.append({
        'chamada': f'MAX_SUBARRAY({inicio}, {fim})',
        'subarray': A[inicio:fim+1],
        'melhor_soma': resposta[2],
        'tipo': resposta[3]
    })

    return resposta[0], resposta[1], resposta[2], contador


def max_subarray_cruzando(A, inicio, meio, fim):
    soma = 0
    melhor_esq = float('-inf')
    indice_esq = meio

    for i in range(meio, inicio - 1, -1):
        soma += A[i]
        if soma > melhor_esq:
            melhor_esq = soma
            indice_esq = i

    soma = 0
    melhor_dir = float('-inf')
    indice_dir = meio + 1

    for j in range(meio + 1, fim + 1):
        soma += A[j]
        if soma > melhor_dir:
            melhor_dir = soma
            indice_dir = j

    return indice_esq, indice_dir, melhor_esq + melhor_dir


## Execução

In [2]:
A = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
inicio, fim, soma, contador = max_subarray(A)

print('Vetor:', A)
print('Início:', inicio)
print('Fim:', fim)
print('Subarray máximo:', A[inicio:fim+1])
print('Soma máxima:', soma)
print('Chamadas:', contador.chamadas)

display(pd.DataFrame(contador.passos))


Vetor: [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Início: 7
Fim: 10
Subarray máximo: [18, 20, -7, 12]
Soma máxima: 43
Chamadas: 31


,chamada,subarray,melhor_soma,tipo
0,"MAX_SUBARRAY(0, 0)",[13],13,caso base
1,"MAX_SUBARRAY(1, 1)",[-3],-3,caso base
2,"MAX_SUBARRAY(0, 1)","[13, -3]",13,esquerda
3,"MAX_SUBARRAY(2, 2)",[-25],-25,caso base
4,"MAX_SUBARRAY(3, 3)",[20],20,caso base
5,"MAX_SUBARRAY(2, 3)","[-25, 20]",20,direita
6,"MAX_SUBARRAY(0, 3)","[13, -3, -25, 20]",20,direita
7,"MAX_SUBARRAY(4, 4)",[-3],-3,caso base
8,"MAX_SUBARRAY(5, 5)",[-16],-16,caso base
9,"MAX_SUBARRAY(4, 5)","[-3, -16]",-3,esquerda


## Kadane

In [3]:
def kadane(A):
    melhor_soma = A[0]
    soma_atual = A[0]
    inicio_atual = 0
    melhor_inicio = 0
    melhor_fim = 0
    passos = []

    for i in range(1, len(A)):
        if A[i] > soma_atual + A[i]:
            soma_atual = A[i]
            inicio_atual = i
        else:
            soma_atual += A[i]

        if soma_atual > melhor_soma:
            melhor_soma = soma_atual
            melhor_inicio = inicio_atual
            melhor_fim = i

        passos.append({
            'i': i,
            'A[i]': A[i],
            'soma_atual': soma_atual,
            'melhor_soma': melhor_soma,
            'melhor_intervalo': f'{melhor_inicio}..{melhor_fim}'
        })

    return melhor_inicio, melhor_fim, melhor_soma, passos


## Execução Kadane

In [4]:
inicio, fim, soma, passos = kadane(A)

print('Subarray máximo:', A[inicio:fim+1])
print('Soma máxima:', soma)

display(pd.DataFrame(passos))


Subarray máximo: [18, 20, -7, 12]
Soma máxima: 43


,i,A[i],soma_atual,melhor_soma,melhor_intervalo
0,1,-3,10,13,0..0
1,2,-25,-15,13,0..0
2,3,20,20,20,3..3
3,4,-3,17,20,3..3
4,5,-16,1,20,3..3
5,6,-23,-22,20,3..3
6,7,18,18,20,3..3
7,8,20,38,38,7..8
8,9,-7,31,38,7..8
9,10,12,43,43,7..10
